<a href="https://colab.research.google.com/github/sonashah02/retail-churn-market-basket-analysis/blob/main/MarketBasketAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [138]:
import pandas as pd
import numpy as np

In [139]:
cleaned_data = pd.read_csv('/content/cleaned_orders1.csv')

In [ ]:
cleaned_data.head()

## More Data Cleaning

In [141]:
# Rename Name to Order ID
cleaned_data = cleaned_data.rename(columns={'Name': 'OrderID'})

In [142]:
# Forward fill customer names and order total
cleaned_data['Billing Name'] = cleaned_data.groupby('OrderID')['Billing Name'].transform('first')
cleaned_data['Total'] = cleaned_data.groupby('OrderID')['Total'].transform('first')

In [143]:
# Make dates datetimes using a function

def make_datetime(df, column):
    df[column] = pd.to_datetime(df[column])
    return df

cleaned_data = make_datetime(cleaned_data, 'Created at')
cleaned_data = make_datetime(cleaned_data,'Fulfilled at')
cleaned_data = make_datetime(cleaned_data,'Cancelled at')
cleaned_data = make_datetime(cleaned_data,'Paid at')

In [144]:
# Rename columns
cleaned_data = cleaned_data.rename(columns={'Created at': 'order_date'})
cleaned_data = cleaned_data.rename(columns={'Billing Name': 'billing_name'})

In [ ]:
# Remove the # before every OrderID
cleaned_data.loc[:, 'OrderID'] = cleaned_data['OrderID'].str.replace('#', '')
display(cleaned_data.head())

In [146]:
# Remove NEW/RETIRED/PREORDER from Lineitem name

cleaned_data.loc[:, 'Lineitem name'] = cleaned_data['Lineitem name'].str.replace('NEW - ', '')
cleaned_data.loc[:, 'Lineitem name'] = cleaned_data['Lineitem name'].str.replace('RETIRED - ', '')
cleaned_data.loc[:, 'Lineitem name'] = cleaned_data['Lineitem name'].str.replace('PREORDER - ', '')

In [174]:
# Only keep lines of orders created 2024-2026

cleaned_data_2024 = cleaned_data[cleaned_data['order_date'].dt.year >= 2024]

## Market Basket Analysis

In [9]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Group the `Lineitem name` by `OrderID` to create a list of items for each transaction (order).

In [10]:
orders_grouped = cleaned_data_2024.groupby('OrderID')['Lineitem name'].apply(list).reset_index()
transactions = orders_grouped['Lineitem name'].tolist()

print(f"Number of transactions: {len(transactions)}")
print("First 5 transactions:")
for i, transaction in enumerate(transactions[:5]):
    print(f"Order {orders_grouped['OrderID'].iloc[i]}: {transaction}")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Number of transactions: 4175
First 5 transactions:
Order 1664: ['Giftology Scottsdale E-Gift Card - $50.00 USD', 'Giftology Scottsdale E-Gift Card - $25.00 USD']
Order 1665: ['A Little Love You Grandma Bracelet - Silver']
Order 1666: ["Lookin' Sharp Cactus & Southwest Kitchen Woven Dish Towel Set of 2", 'Stemless Wine Glass - Just Chill', 'Potted Cactus Heart Trinket Tray', 'Four Shot Glass Aqua Cactus Set - Boxed']
Order 1667: ['Hello Gorgeous! Lipstick', 'Here Comes the Sun', 'Party Hat - Hats off to 20 Years!', 'Love Blooms Here Flower Box', 'One Cool Chick - Chick in Egg']
Order 1668: ['Hello Gorgeous! Lipstick', 'Party Hat - Hats off to 20 Years!']


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Next, use `TransactionEncoder` from `mlxtend` to transform the list of transactions into a one-hot encoded DataFrame. This is necessary for the Apriori algorithm.

In [11]:
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_transactions = pd.DataFrame(te_ary, columns=te.columns_)

print("Shape of one-hot encoded DataFrame:", df_transactions.shape)
display(df_transactions.head())

Shape of one-hot encoded DataFrame: (4175, 3173)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

,"Beach, please!",Bear with Me,Cheery Cherries,Clementine Orange - Cutie Pie,Creative Quill - Feather,Eggs and Bacon - wakey wakey,Fried Egg and Bacon - wakey wakey,"Hot Air Balloon - Red, White & Blue",Mah-jical Mahjong Tile,Orange - Cutie Pie,...,hydraAROMATHERAPY Lifestyle Shower Burst Variety Pack,hydraAROMATHERAPY Shower Burst Duo in Escape,hydraAROMATHERAPY Shower Burst Duo in Headache Buster,hydraAROMATHERAPY Shower Burst Duo in Refresh,hydraAROMATHERAPY Shower Burst Duo in Relax,hydraAROMATHERAPY Shower Burst Duo in Rise,hydraAROMATHERAPY Shower Burst Duo in Stress Buster,hydraAROMATHERAPY Shower Burst Duo in Unwind,hydraAROMATHERAPY Signature Shower Burst Variety Pack,hydraAROMATHERAPY Trio in Morning Shower Burst Variety Pack
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

### Applying the Apriori Algorithm

Now that we have our one-hot encoded transaction data, we can apply the Apriori algorithm to find frequent itemsets. The `min_support` parameter specifies the minimum support threshold for an itemset to be considered frequent.

In [12]:
frequent_itemsets = apriori(df_transactions, min_support=0.002, use_colnames=True)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(lambda x: len(x))

print("Frequent Itemsets found (with min_support=0.002):")
display(frequent_itemsets.head(15))

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Frequent Itemsets found (with min_support=0.002):


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,support,itemsets,length
0,0.002156,( Clementine Orange - Cutie Pie),1
1,0.004790,( Creative Quill - Feather),1
2,0.002156,( Eggs and Bacon - wakey wakey),1
3,0.007186,"( Hot Air Balloon - Red, White & Blue)",1
4,0.006228,( Oyster - Shuck yeah! mini),1
5,0.006467,( Peacock - Fancy Feathers mini),1
6,0.002156,( Porch Pal Goose),1
7,0.004072,( Soccer Ball - Kickin' it Mini),1
8,0.005988,( Tote - Tote-ally Cute mini),1
9,0.002156,(4-Piece Cactus Cheese Spreaders Set),1


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

### Generating Association Rules

Now we will generate association rules from the frequent itemsets. We will use `association_rules` and set metrics like `min_threshold` to filter interesting rules.

In [13]:
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)
rules = rules.sort_values(['lift'], ascending=False)

print("Association Rules found (sorted by Lift):")
display(rules.head())

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Association Rules found (sorted by Lift):


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
1,(April Dishcloth Set of 3 by Geometry),(April Kitchen Tea Towel by Geometry),0.003353,0.002874,0.002156,0.642857,223.660714,1.0,0.002146,2.791952,0.998878,0.529412,0.641828,0.696429
0,(April Kitchen Tea Towel by Geometry),(April Dishcloth Set of 3 by Geometry),0.002874,0.003353,0.002156,0.750000,223.660714,1.0,0.002146,3.986587,0.998399,0.529412,0.749159,0.696429
13,(Double Sided - Lemon Bliss Kitchen Tea Towel ...,(Lemon Waves Dishcloth Set of 3 by Geometry),0.005269,0.005269,0.002156,0.409091,77.634298,1.0,0.002128,1.683390,0.992348,0.257143,0.405961,0.409091
12,(Lemon Waves Dishcloth Set of 3 by Geometry),(Double Sided - Lemon Bliss Kitchen Tea Towel ...,0.005269,0.005269,0.002156,0.409091,77.634298,1.0,0.002128,1.683390,0.992348,0.257143,0.405961,0.409091
20,(Power Mist Frosted Mint Hand Sanitizer - 1 fl...,(Power Mist Pure Lavender Hand Sanitizer - 1 f...,0.005749,0.005988,0.002395,0.416667,69.583333,1.0,0.002361,1.704021,0.991327,0.256410,0.413153,0.408333


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

### Interpreting and Filtering Association Rules

To focus on the most meaningful relationships, we can filter the association rules based on metrics like `confidence` and `lift`.

*   **Confidence**: Indicates the likelihood that item Y is purchased when item X is purchased. A high confidence suggests a strong association.
*   **Lift**: Measures how much more likely item Y is purchased when item X is purchased, relative to its baseline probability. A lift greater than 1 suggests a positive correlation, while a lift significantly greater than 1 indicates a strong association.

In [14]:
filtered_rules = rules[(rules['confidence'] > 0.1) & (rules['lift'] > 1.0)]
filtered_rules = filtered_rules.sort_values('lift', ascending=False)

print("Filtered Association Rules (Confidence > 0.1, Lift > 1.0):")
display(filtered_rules)

if filtered_rules.empty:
    print("No association rules found with confidence > 0.1 and lift > 1.0. Consider adjusting thresholds further.")

Filtered Association Rules (Confidence > 0.1, Lift > 1.0):


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
1,(April Dishcloth Set of 3 by Geometry),(April Kitchen Tea Towel by Geometry),0.003353,0.002874,0.002156,0.642857,223.660714,1.0,0.002146,2.791952,0.998878,0.529412,0.641828,0.696429
0,(April Kitchen Tea Towel by Geometry),(April Dishcloth Set of 3 by Geometry),0.002874,0.003353,0.002156,0.750000,223.660714,1.0,0.002146,3.986587,0.998399,0.529412,0.749159,0.696429
13,(Double Sided - Lemon Bliss Kitchen Tea Towel ...,(Lemon Waves Dishcloth Set of 3 by Geometry),0.005269,0.005269,0.002156,0.409091,77.634298,1.0,0.002128,1.683390,0.992348,0.257143,0.405961,0.409091
12,(Lemon Waves Dishcloth Set of 3 by Geometry),(Double Sided - Lemon Bliss Kitchen Tea Towel ...,0.005269,0.005269,0.002156,0.409091,77.634298,1.0,0.002128,1.683390,0.992348,0.257143,0.405961,0.409091
20,(Power Mist Frosted Mint Hand Sanitizer - 1 fl...,(Power Mist Pure Lavender Hand Sanitizer - 1 f...,0.005749,0.005988,0.002395,0.416667,69.583333,1.0,0.002361,1.704021,0.991327,0.256410,0.413153,0.408333
21,(Power Mist Pure Lavender Hand Sanitizer - 1 f...,(Power Mist Frosted Mint Hand Sanitizer - 1 fl...,0.005988,0.005749,0.002395,0.400000,69.583333,1.0,0.002361,1.657086,0.991566,0.256410,0.396531,0.408333
14,"(Fun Doh 4"" Mystery Dumpling Squishies)","(Fun Doh Squishy Mystery Mini Petite 2.5"" Dump...",0.011737,0.006228,0.003114,0.265306,42.602041,1.0,0.003041,1.352635,0.988124,0.209677,0.260702,0.382653
15,"(Fun Doh Squishy Mystery Mini Petite 2.5"" Dump...","(Fun Doh 4"" Mystery Dumpling Squishies)",0.006228,0.011737,0.003114,0.500000,42.602041,1.0,0.003041,1.976527,0.982646,0.209677,0.494062,0.382653
17,(Geometry Monogram Floral A Dishcloth Set of 3...,(Greetings from Arizona Kitchen Tea Towel by G...,0.003593,0.028743,0.002395,0.666667,23.194444,1.0,0.002292,2.913772,0.960337,0.080000,0.656802,0.375000
18,(Greetings from Arizona Kitchen Tea Towel by G...,(Greetings from Arizona Dishcloth Set of 3 by ...,0.028743,0.011018,0.007186,0.250000,22.690217,1.0,0.006869,1.318643,0.984217,0.220588,0.241644,0.451087


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

### Listing Specific Item Relationships for Business Action

To make the insights from the association rules directly actionable, here's a list of the top rules, detailing which items tend to be purchased together. This can be directly used for marketing, product placement, and bundle offers.

In [16]:
print("\n--- Top Association Rules (Items Commonly Bought Together) ---")
# Iterate through the filtered_rules and print actionable insights
for index, row in filtered_rules.head(50).iterrows(): # Displaying top 10 rules for brevity
    antecedent_items = ', '.join(list(row['antecedents']))
    consequent_items = ', '.join(list(row['consequents']))

    print(f"\nIf a customer buys '{antecedent_items}', they are likely to also buy '{consequent_items}'.")
    print(f"  - Confidence: {row['confidence']:.2f} (This happens in {row['confidence']:.2%}"\
          f" of transactions containing '{antecedent_items}')")
    print(f"  - Lift: {row['lift']:.2f} (This co-occurrence is {row['lift']:.2f} times more likely than by chance)")
    print(f"  - Support: {row['support']:.4f} (Both items appear together in {row['support']:.2%} of all transactions)")

if filtered_rules.empty:
    print("No specific item relationships found based on current thresholds. Consider adjusting min_support, min_confidence, or min_lift.")
else:
    print("\nThese insights highlight specific product combinations that you can leverage for cross-selling, bundling, and targeted promotions.")


--- Top Association Rules (Items Commonly Bought Together) ---

If a customer buys 'April Dishcloth Set of 3 by Geometry', they are likely to also buy 'April Kitchen Tea Towel by Geometry'.
  - Confidence: 0.64 (This happens in 64.29% of transactions containing 'April Dishcloth Set of 3 by Geometry')
  - Lift: 223.66 (This co-occurrence is 223.66 times more likely than by chance)
  - Support: 0.0022 (Both items appear together in 0.22% of all transactions)

If a customer buys 'April Kitchen Tea Towel by Geometry', they are likely to also buy 'April Dishcloth Set of 3 by Geometry'.
  - Confidence: 0.75 (This happens in 75.00% of transactions containing 'April Kitchen Tea Towel by Geometry')
  - Lift: 223.66 (This co-occurrence is 223.66 times more likely than by chance)
  - Support: 0.0022 (Both items appear together in 0.22% of all transactions)

If a customer buys 'Double Sided - Lemon Bliss Kitchen Tea Towel by Geometry', they are likely to also buy 'Lemon Waves Dishcloth Set of 3 b

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

### Summary of Market Basket Analysis

Based on the frequent itemsets and association rules, we can identify patterns of products that are commonly purchased together. These insights can be valuable for:

*   **Product Placement**: Strategically placing associated items together in a physical or online store.
*   **Promotional Bundles**: Creating product bundles or 'buy one, get one' offers for related items.
*   **Recommendation Systems**: Improving product recommendation engines.
*   **Inventory Management**: Ensuring adequate stock of frequently co-purchased items.

**Key Findings (based on filtered rules with confidence > 0.3, Lift > 1.5):**

*   The rules indicate strong relationships between specific items. For instance, if `Cacti Kitchen Tea Towel by Geometry` is in the basket, there's a high probability (`0.879998` confidence) that `Positive Vibes Dishcloth Set of 3 by Geometry` is also purchased, and vice-versa, as indicated by a high `lift` value of `18.651761`. This suggests these items are almost always bought together when they appear in a transaction.

This analysis provides actionable insights into customer purchasing behavior, which can be leveraged for various business strategies.